# A2-Lite — one orientation-rebalanced geometry teaching pilot
Latest revision: **A2-LITE-512-ANGLE001-KD-2026-10-14-r1**

Question: does reducing the largest teacher gradient preserve detection and improve the student?

Only one objective coefficient changes from the reviewed geometry-only run: **teacher orientation multiplier 1.0 → 0.01**. Teacher depth and size multipliers remain 1.0. The overall teacher coefficient still ramps to 0.25 over 128 updates, so the maximum effective orientation coefficient is 0.0025. Do not renormalize the teacher sum. All ground-truth losses, including orientation, remain unchanged.

Same A2 teacher, A2-Lite architecture, 640×192 input, 50 queries, original pruned weights, exact 512 TRAIN frames, seed, matching, class-balanced teacher means, temperature 2, eligibility masks, optimizer, and five-epoch budget. Both direct teacher classification and box losses remain disabled. Frame groups are balanced; object counts are not.

The reviewed 32-frame gradient audit motivates this test but does not prove the cause of the pedestrian regression. Its two checkpoints are evidence only: neither initializes this run. Reuse the original 512-frame teacher cache; no teacher inference or historical reference training is repeated.

Predeclared benefit guardrail versus label-only: all five AP metrics nonregressing, mean moderate 3D AP gain ≥0.1 point, and nearby recall drop ≤2 absolute percentage points for either class. Evaluate all 3,769 VAL frames at fixed epoch 5; do not select a best checkpoint. Also report changes versus all-head and original geometry-only KD. This is a single-seed historical comparison, not a simultaneous controlled repeat. Relative benefit does not qualify deployment.

A100 preferred. Keep the current working session if available. No phone, native extension build, old CUDA runtime, Torch replacement, or Core ML conversion. Ordinary packages must match the historical runs. Optional input archive restore accelerates staging when the separately prepared cache exists; otherwise four-worker staging reuses verified local data.

Run sections 1–8 in order. After a reset, rerun 1–5, then resume 6. Save every 128 updates; up to 127 interrupted updates can repeat. Return the ZIP even if preflight or a quality check fails. No full-data training follows automatically.

## 1. Drive, all paths and helpers
Edit paths here if needed. AUDIT can be the completed gradient folder or its results ZIP. PILOT must be the ORIGINAL 512-frame recovery folder or ZIP, not the full-data or geometry run.


In [ ]:
# A2-LITE-512-ANGLE001-KD-2026-10-14-r1
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, re, shlex, subprocess, sys, zipfile
REVISION='A2-LITE-512-ANGLE001-KD-2026-10-14-r1'
RUN_ID='a2_lite_512_angle001_kd_r1'
if not re.fullmatch(r'[a-z0-9_]+',RUN_ID):raise ValueError('Use lowercase letters, digits and underscores')
MOBILE_REPO=Path('/content/mobile_adas3d')
AUDIT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/diagnostics/a2_lite_geometry_gradient/a2_lite_geometry_gradient_r1')
PILOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_recovery/a2_lite_640_recovery_kd_r1')
INITIALIZATION=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite/a2_lite_e1_d1_cpu_screen_r1/a2_lite_initialization.pth')
COMPACT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_resolution/a2_lite_resolution_cpu_screen_r1')
SPLITS=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
SOURCE_REPO=Path('/content/MonoDETR_A2LITE_ANGLE001_'+RUN_ID)
DATASET=Path('/content/kitti_a2_lite_accuracy_angle001_'+RUN_ID)
# Optional reusable input archive. Do not prepare it in this GPU run.
CACHE=Path('/content/drive/MyDrive/mobile_adas3d_inputs/kitti_chen_a2_v1')
LOCAL_ARCHIVE=Path('/content/a2_lite_archive_local/kitti_chen_7481.zip')
CANDIDATES=[Path(p) for p in (
    '/content/kitti_a2_lite_accuracy_geometry_a2_lite_512_geometry_kd_r1',
    '/content/kitti_a2_lite_accuracy_recovery_a2_lite_640_recovery_kd_r1',
    '/content/kitti_a2_lite_accuracy_full_a2_lite_640_full_recovery_kd_r1',
    '/content/kitti_a2_lite_accuracy_cache_prepare_v1',
    '/content/kitti_a2_lite_accuracy_a2_lite_640_accuracy_r1',
    '/content/kitti_m68','/content/kitti_m67','/content/kitti',
    '/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti')]
OUTPUT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_balanced_geometry')/RUN_ID
PYTHON=Path(sys.executable);SCRIPT=MOBILE_REPO/'scripts/recover_a2_lite_balanced_geometry.py'
LOGS=OUTPUT/'colab_logs';LOGS.mkdir(parents=True,exist_ok=True)
READY=False;TINY_PASSED=False;KD_READY=False
def run_logged(command,name,cwd=None):
    command=list(map(str,command));log_path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',log_path,flush=True);tail=deque(maxlen=40)
    with log_path.open('w') as log:
        log.write(shlex.join(command)+'\n');log.flush()
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
        finally:process.stdout.close()
        status=process.wait();log.write(f'\nEXIT CODE: {status}\n');log.flush()
    if status:raise RuntimeError(f'Exit {status}; full log={log_path}\n'+'\n'.join(tail))
    return log_path
def stage(name,*extra):
    if not READY and name!='bundle':raise RuntimeError('Run sections 1 and 2 first')
    return run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,name,'--output',OUTPUT,'--device','cuda',*extra],name,MOBILE_REPO)
print(REVISION,'\nTeacher multipliers: depth=1, size=1, orientation=0.01. All GT unchanged.\nOutput:',OUTPUT)


## 2. Update main and verify the existing runtime
Pull before checking helper files. Check Python/Torch compatibility before ordinary dependency installation. Do not restore a historical GPU stack or overwrite a saved manifest to force a comparison.


In [ ]:
READY=False
if not MOBILE_REPO.exists():run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty:raise RuntimeError('Preserve tracked edits and use clean main.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main':raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only'],'update',MOBILE_REPO)
if not SCRIPT.is_file():raise FileNotFoundError('Pull the published orientation-rebalanced notebook and helpers from main')
for path in (AUDIT,PILOT,INITIALIZATION,COMPACT,SPLITS/'train.txt',SPLITS/'val.txt'):
    if not path.exists():raise FileNotFoundError(f'{path}: correct section 1 paths')
if PILOT.is_dir():saved=json.loads((PILOT/'recovery_manifest.json').read_text())['cpu_environment']
else:
    with zipfile.ZipFile(PILOT) as archive:
        names=[n for n in archive.namelist() if n=='recovery_manifest.json' or n.endswith('/recovery_manifest.json')]
        if len(names)!=1:raise RuntimeError('Expected one original 512-frame pilot manifest')
        saved=json.loads(archive.read(names[0]))['cpu_environment']
probe=("import sys,json,torch; saved=json.loads(sys.argv[1]); "
       "actual={'python':'.'.join(map(str,sys.version_info[:2])),'torch':torch.__version__.split('+')[0]}; "
       "changes={k:{'saved':saved[k],'current':v} for k,v in actual.items() if saved[k]!=v}; "
       "assert not changes,'Stop for fresh matched-reference review; do not restore old CUDA: '+json.dumps(changes)")
run_logged([PYTHON,'-c',probe,json.dumps(saved)],'reference_python_torch')
packages=[name+'=='+saved[name] for name in ('numpy','timm','onnxruntime','pillow','scipy','opencv-python-headless')]+['pyyaml','scikit-image']
run_logged([PYTHON,'-m','pip','install','-q',*packages],'ordinary_dependencies')
run_logged([PYTHON,'-c',"import torch;assert torch.cuda.is_available(),'Select a GPU runtime';torch.cuda.init();print(torch.__version__,torch.cuda.get_device_name(0),'portable training; no native attention extension')"],'gpu_probe')
READY=True


## 3. Tests, reviewed audit and efficient staging
Verify the approved audit before staging. If a completed reusable archive exists, copy it once and extract locally, verifying every reviewed input hash. A corrupt completed cache stops instead of silently falling back.

Without a completed archive, use four parallel copy workers with verified local reuse and progress. CPU preparation of the reusable archive is optional in a separate session; do not switch this live GPU session to CPU and back. The model prepare stage still checks all 7,481 input triplets and the exact original 512-frame TRAIN subset.


In [ ]:
if not READY:raise RuntimeError('Run sections 1 and 2 first')
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_balanced_geometry.py','-v'],'balanced_tests',MOBILE_REPO)
probe=("import sys;from pathlib import Path;sys.path.insert(0,sys.argv[1]); "
       "from recover_a2_lite_balanced_geometry import reviewed_audit; "
       "m,comparison,audit,records=reviewed_audit(Path(sys.argv[2])); "
       "print('Approved zero-update audit verified; original training subset:',len(m['pilot_ids']))")
run_logged([PYTHON,'-c',probe,MOBILE_REPO/'scripts',AUDIT],'review_gradient_evidence',MOBILE_REPO)
extra=[]
for candidate in CANDIDATES:extra += ['--candidate',candidate]
if (CACHE/'kitti_input_cache.json').is_file():
    if not (CACHE/'kitti_chen_7481.zip').is_file():raise RuntimeError('Completed cache receipt exists but archive is missing; preserve cache and review')
    print('Using completed reusable input archive; no per-image Drive copying.')
    run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_input_cache.py','-v'],'input_cache_tests',MOBILE_REPO)
    run_logged([PYTHON,'-u',MOBILE_REPO/'scripts/cache_a2_lite_inputs.py','restore',
                '--pilot',PILOT,'--cache',CACHE,'--dataset',DATASET,'--local-archive',LOCAL_ARCHIVE],'restore_input_archive',MOBILE_REPO)
else:
    print('No completed input archive; preserving any incomplete cache and using four-worker verified staging.')
    run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_data_staging.py','-v'],'staging_tests',MOBILE_REPO)
    run_logged([PYTHON,'-u',MOBILE_REPO/'scripts/stage_a2_lite_data.py',
                '--pilot',PILOT,'--dataset',DATASET,'--workers','4',*extra],'parallel_staging',MOBILE_REPO)
stage('prepare','--audit',AUDIT,'--pilot',PILOT,'--initialization',INITIALIZATION,
      '--compact',COMPACT,'--repo',SOURCE_REPO,'--dataset',DATASET,'--split-dir',SPLITS,*extra)
MANIFEST=json.loads((OUTPUT/'geometry_manifest.json').read_text())
assert MANIFEST['recipe']['teacher_component_weights']=={'dimensions':1.0,'depth':1.0,'angle':0.01}
assert len(MANIFEST['pilot_ids'])==512 and MANIFEST['recipe']['epochs']==5
print('New manifest:',MANIFEST['revision'],'— fixed updates:',512*5,'— GT angle coefficient:',MANIFEST['recipe']['gt_weights']['loss_angle'])


## 4. Initialization parity and disposable tiny fit
Reproduce the original 640×192 fixtures and run the unchanged 16-frame, 128-update label-only learnability check. Its checkpoint NEVER initializes the pilot. A failed tiny fit stops training; return the bundle rather than lowering the gate.


In [ ]:
stage('smoke')
stage('tiny')
TINY=json.loads((OUTPUT/'tiny_fit.json').read_text());TINY_PASSED=bool(TINY['passed'])
print(json.dumps({k:TINY[k] for k in ('passed','gt_loss_ratio','geometry_error_ratio','geometry_ratio_per_class','evaluation_scope')},indent=2))
if not TINY_PASSED:print('STOP: return section 8 evidence; do not lower the gate.')


## 5. Reuse the original teacher cache and check the new weighted gradients
No new teacher inference. Verify all 512 original TRAIN teacher records against their hashes and current encoded ground truth. Check finite nonzero depth, size and orientation gradients, unchanged buffers, zero direct classification/box teacher counts, and zero optimizer steps.


In [ ]:
KD_READY=False
if TINY_PASSED:
    stage('cache','--pilot',PILOT)
    CACHE_REPORT=json.loads((OUTPUT/'teacher_cache_complete.json').read_text())
    print(json.dumps({k:CACHE_REPORT[k] for k in ('images','encoded_gt_counts','qualified_teacher_matches','kd_pilot_authorized','teacher_inference_performed')},indent=2))
    if CACHE_REPORT['kd_pilot_authorized']:
        stage('kd-smoke')
        CHECK=json.loads((OUTPUT/'geometry_kd_smoke.json').read_text());KD_READY=bool(CHECK['passed'])
        assert CHECK['optimizer_steps']==0 and CHECK['buffers_unchanged']
        print('Teacher multipliers:',MANIFEST['recipe']['teacher_component_weights'])
        print(json.dumps({k:CHECK[k] for k in ('passed','active_counts','component_gradient_abs_sum','optimizer_steps')},indent=2))
    else:print('STOP: teacher cache coverage did not qualify; return section 8 evidence.')
else:print('Skipped: tiny learnability check failed.')


## 6. Train ONE new arm — five fixed epochs
2,560 updates, batch size 1, original pruned weights and unchanged deterministic frame order. All ground-truth losses stay on. The teacher loss is size + depth + 0.01×orientation, multiplied by the unchanged overall ramp to 0.25.

Same learning rates, AdamW, clipping, frozen BatchNorm and early backbone, no augmentation/dropout/AMP. The new kd/ folder contains only this rebalanced run. Do not change coefficients or RUN_ID midway. After interruption, rerun sections 1–5 and resume here.


In [ ]:
if KD_READY:
    stage('train')
else:print('No rebalanced training started; return the preflight evidence.')


## 7. Evaluate fixed epoch 5 and compare all four recipes
Run all 3,769 VAL images once at the declared endpoint. Compare with the reviewed label-only, all-head KD and original geometry-only endpoints; only this new arm is retrained. Report both AP and nearby recall. Higher AP cannot hide a pedestrian or vehicle recall drop beyond the existing guardrail. No automatic full-data training or phone deployment.


In [ ]:
if KD_READY:
    stage('evaluate')
    stage('compare')
    REPORT=json.loads((OUTPUT/'balanced_geometry_comparison.json').read_text())
    for name,value in REPORT['results'].items():
        print(name,json.dumps({'metrics':value['metrics'],
                              'nearby_recall':{c:value['nearby']['classes'][c]['near_recall'] for c in ('Vehicle','Pedestrian')}},indent=2))
    print(json.dumps({k:REPORT[k] for k in ('teacher_component_weights','comparisons','benefit_guardrail_passed',
                                          'comparison_limit','full_training_authorized','deployment_qualified')},indent=2))
else:print('Validation skipped: preflight did not qualify training.')


## 8. Download results — also after a failure
Return a2_lite_balanced_geometry_results.zip. Includes the reviewed audit proof, metrics, predictions, teacher-cache identities and combined logs; excludes large weights, model packages and images. Checkpoints remain on Drive for resume. No phone is needed yet.


In [ ]:
BUNDLE=OUTPUT/'a2_lite_balanced_geometry_results.zip'
try:
    if not SCRIPT.is_file():raise RuntimeError('Helper missing; package readable logs directly')
    stage('bundle')
except Exception as error:
    print('Bundle helper unavailable:',error,'— packaging readable results/logs directly.')
    with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and 'assets' not in path.relative_to(OUTPUT).parts and path.suffix in ('.json','.csv','.txt','.log'):
                archive.write(path,path.relative_to(OUTPUT))
print(BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
